In [ ]:
import os

from  dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from IPython.display import Image, display

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",  
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# Tool Calls
- Agent 支持静态和动态绑定工具，后者需要用到中间件
- 内置工具：https://docs.langchain.com/oss/python/integrations/tools

## 1. Custom Tools

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from rich import print as rprint


class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，例如北京、上海、广州")
    include_forecast: bool = Field(
        default=False,
        description="是否查询未来几天的天气预报。如果用户问'这几天'、'未来几天'，则为 True"
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, include_forecast: bool = False) -> dict:
    """获取指定城市的天气信息。"""
    result = {
        "city": city,
        "temperature": "25°C",
        "condition": "晴天",
    }

    if include_forecast:
        result["forecast"] = [
            {"day": "Monday", "temperature": "26°C", "condition": "多云"},
            {"day": "Tuesday", "temperature": "24°C", "condition": "小雨"},
            {"day": "Wednesday", "temperature": "27°C", "condition": "晴天"},
        ]

    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="你是一个天气查询助手，根据用户的提问查询天气信息。如果问题与天气无关，请礼貌地告知用户。",
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="北京这几天的天气怎么样？")
    ]
})
display(Image(agent.get_graph().draw_mermaid_png())) # 可视化
rprint(response)

## 2. Built-in Tools

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(".env")


class SearchInput(BaseModel):
    query: str = Field(..., description="需要搜索验证的问题或关键词")


class NobelWinner(BaseModel):
    name: str = Field(..., description="得主姓名或组织名称")
    category: str = Field(..., description="诺贝尔奖类别，例如物理学、化学、生理学或医学、文学、和平奖、经济学")
    nationality: str = Field(..., description="国籍；如果是组织则写其所属国家/地区或'组织'")
    achievements: list[str] = Field(..., description="获奖原因或相关成就")


class NobelAnswer(BaseModel):
    year: int = Field(..., description="年份")
    winners: list[NobelWinner] = Field(..., description="该年度诺贝尔奖得主列表")
    sources: list[str] = Field(..., description="搜索验证所依据的来源链接")


tavily = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=5,
    search_depth="basic",
    topic="general",
    include_answer=True,
)

model_with_tools = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)


@tool(args_schema=SearchInput)
def web_search(query: str) -> str:
    """联网搜索并返回用于事实核验的结果。"""
    result = tavily.invoke({"query": query})
    return json.dumps(result, ensure_ascii=False)


system_prompt = """
你是一个专业智能助手。
回答任何涉及事实、时效或名单的问题前，必须先调用 web_search 工具进行搜索验证。
回答时只使用搜索结果中能验证的信息。
如果搜索结果不足以确认，请明确说明无法确认。
"""

agent = create_agent(
    model=model_with_tools,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=ToolStrategy(NobelAnswer),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="2025年诺贝尔奖得主有哪些？请按奖项类别列出。")
    ]
})

rprint(response["structured_response"])

## 3. Multiple Tools

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from IPython.display import Image, display
from rich import print as rprint
import random


# 工具 1：天气查询
class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，例如北京、上海、东京、纽约")


def _random_temp(base_min: int, base_max: int) -> str:
    """生成随机温度，范围基于城市常年气温。"""
    return f"{random.randint(base_min, base_max)}°C"


@tool(args_schema=WeatherInput)
def get_weather(city: str) -> dict:
    """查询指定城市的实时天气信息（模拟数据，温度为随机生成）。"""
    weather_db = {
        "北京": {"temperature": _random_temp(20, 38), "condition": "晴", "humidity": "45%"},
        "上海": {"temperature": _random_temp(25, 40), "condition": "多云", "humidity": "70%"},
        "东京": {"temperature": _random_temp(18, 35), "condition": "小雨", "humidity": "80%"},
        "纽约": {"temperature": _random_temp(15, 35), "condition": "晴", "humidity": "50%"},
        "伦敦": {"temperature": _random_temp(10, 28), "condition": "阴", "humidity": "65%"},
        "新加坡": {"temperature": _random_temp(26, 35), "condition": "雷阵雨", "humidity": "85%"},
    }
    return weather_db.get(city, {"temperature": "N/A", "condition": "暂无数据", "humidity": "N/A"})


# 工具 2：汇率查询
class ExchangeInput(BaseModel):
    from_currency: str = Field(description="源货币代码（三位大写），例如 USD、CNY、JPY、EUR")
    to_currency: str = Field(description="目标货币代码（三位大写），例如 USD、CNY、JPY、EUR")


@tool(args_schema=ExchangeInput)
def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    """查询两种货币之间的实时汇率（模拟数据）。"""
    rates = {
        ("USD", "CNY"): 7.25,
        ("USD", "JPY"): 150.50,
        ("USD", "EUR"): 0.92,
        ("CNY", "USD"): 0.138,
        ("CNY", "JPY"): 20.76,
        ("CNY", "EUR"): 0.127,
        ("JPY", "CNY"): 0.048,
        ("JPY", "USD"): 0.0066,
        ("EUR", "USD"): 1.09,
        ("EUR", "CNY"): 7.88,
    }
    key = (from_currency.upper(), to_currency.upper())
    if key in rates:
        return {"from": from_currency.upper(), "to": to_currency.upper(), "rate": rates[key]}
    return {"error": f"暂不支持 {from_currency} → {to_currency} 的汇率查询"}


# 工具 3：数学计算
class CalcInput(BaseModel):
    expression: str = Field(description="数学表达式，例如 '1000 * 7.25'、'(100 + 200) / 3'")


@tool(args_schema=CalcInput)
def calculate(expression: str) -> dict:
    """执行四则运算和常用数学函数（sqrt, log, sin, cos 等）。"""
    import math
    safe_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    try:
        result = eval(expression, {"__builtins__": {}}, safe_names)
        return {"expression": expression, "result": round(result, 4)}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


# 创建 Agent（一次性绑定三个自定义工具）
agent = create_agent(
    model=model,
    tools=[get_weather, get_exchange_rate, calculate],
    system_prompt=(
        "你是一个多功能智能助手，可以查询天气、汇率，并进行数学计算。\n"
        "规则：\n"
        "1. 遇到需要多个工具配合的任务（如'查汇率后算钱'），依次调用相关工具\n"
        "2. 遇到独立的子任务（如'查A城市天气 + 查B货币汇率'），可以并行调用\n"
        "3. 回答时用清晰的中文整合所有工具返回的结果"
    ),
)

# 可视化 Agent 的工具调用图
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
# 场景 1：并行调用多个工具（两个独立查询，互不依赖）
print("【场景 1】同时查询天气和汇率（互不依赖 → Agent 可并行调用）")
print("-" * 60)

response1 = agent.invoke({
    "messages": [
        HumanMessage(content="帮我查一下东京的天气，以及现在美元兑日元的汇率是多少？")
    ]
})
rprint(response1)

In [ ]:
# 场景 2：串行调用多个工具（汇率 → 计算，后者依赖前者输出）
print("\n\n【场景 2】先查汇率再算金额（后续工具依赖前一个工具的输出 → Agent 串行调用）")
print("-" * 60)

response2 = agent.invoke({
    "messages": [
        HumanMessage(content="我有2000美元，想换成人民币，能换多少？顺便告诉我上海天气如何。")
    ]
})
rprint(response2)

## 4. Common Issues

### 4.1 How Agents Select Tools

**原理**：Agent 本身不"选择"工具——它将工具的名称、描述和参数 schema 发送给 LLM，由 LLM 根据用户问题决定是否调用、调用哪个工具。LLM 返回的不是普通文本，而是一个 `tool_call` 指令（包含工具名和参数），Agent 框架再负责执行该工具。

**选择流程**：
1. Agent 将 `tools` 列表转换为 OpenAI-compatible function definitions 发给模型
2. 模型根据用户 query + 工具描述做语义匹配，决定是否发起 `tool_call`
3. 如果模型返回 `tool_call`，Agent 执行工具并将结果追加到消息历史
4. 模型再次收到工具结果，决定继续调用其他工具 or 生成最终回答

**关键影响因素**：
- 工具**描述文字**是模型判断的核心依据——描述越清晰、越聚焦，选择越准确
- 参数 schema（Pydantic `Field(description=...)`）帮助模型正确填充参数
- System prompt 中的行为指令会影响模型调用工具的倾向

**最佳实践**：在工具 docstring 中用**动词 + 对象 + 场景**的格式写明工具的用途。

In [ ]:
# 案例：观察 Agent 根据用户问题选择不同工具
from langchain_core.messages import HumanMessage
from rich import print as rprint

# 创建三个职责清晰的工具
@tool
def search_knowledge_base(query: str) -> str:
    """搜索公司内部知识库，查找技术文档、API 说明、架构设计等内部资料。"""
    return f"知识库结果: {query} 相关的 API 文档已找到。"

@tool
def search_public_web(query: str) -> str:
    """搜索公开互联网，查找新闻、行业动态、竞品信息等公开资料。"""
    return f"网络搜索结果: {query} 的最新新闻已找到。"

@tool
def query_database(sql: str) -> str:
    """执行 SQL 查询数据库中的用户订单、交易记录等结构化数据。仅用于内部数据查询。"""
    return "数据库结果: 查询到 42 条记录。"


select_agent = create_agent(
    model=model,
    tools=[search_knowledge_base, search_public_web, query_database],
    system_prompt="你是一个助手，根据用户问题选择最合适的工具。",
)

# 测试 1：内部技术问题 → 应选 search_knowledge_base
print("search - 内部技术问题")
resp = select_agent.invoke({"messages": [HumanMessage(content="用户认证模块的架构设计文档在哪？")]})
for msg in resp["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → use tool: {tc['name']}")

# 测试 2：外部新闻 → 应选 search_public_web
print("\nsearch - 外部新闻")
resp = select_agent.invoke({"messages": [HumanMessage(content="最近AI行业有什么大新闻？")]})
for msg in resp["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → use tool: {tc['name']}")

# 测试 3：数据库查询 → 应选 query_database  
print("\nsearch - 数据查询")
resp = select_agent.invoke({"messages": [HumanMessage(content="上个月有多少笔订单？")]})
for msg in resp["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → use tool: {tc['name']}")

### 4.2 Why Agents Don't Call Tools

**常见原因**：

| # | 原因 | 说明 |
|---|------|------|
| 1 | 工具描述不清晰 | LLM 无法从描述中判断该工具能解决用户的问题 |
| 2 | System prompt 冲突 | prompt 中可能暗示"直接回答"而非"使用工具" |
| 3 | 模型能力不足 | 部分小模型对 function calling 支持较弱，容易忽略工具 |
| 4 | 用户问题与工具无关 | 用户问的内容确实不需要工具——这不是 bug |
| 5 | 参数 schema 过于严格 | 必填字段缺少合理的 `default`，模型无法构造合法参数而放弃调用 |

**演示：坏描述 vs 好描述**

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint


# ---------- ❌ 坏示例：描述太模糊，模型不知道什么时候用 ----------
@tool
def lookup(x: str) -> str:
    """查询信息。"""
    return f"结果: {x}"


# ---------- ✅ 好示例：描述清晰，模型能准确判断使用时机 ----------
class PriceInput(BaseModel):
    product_name: str = Field(description="商品名称，例如 iPhone 16、MacBook Pro")


@tool(args_schema=PriceInput)
def query_product_price(product_name: str) -> dict:
    """查询指定商品的最新市场价格。当用户询问某款产品的价格、多少钱、贵不贵时使用此工具。"""
    price_db = {
        "iphone 16": {"price": 5999, "currency": "CNY"},
        "macbook pro": {"price": 14999, "currency": "CNY"},
    }
    return price_db.get(product_name.lower(), {"error": "未找到该商品"})


# 对比测试
agent_bad = create_agent(model=model, tools=[lookup],
    system_prompt="你是一个助手，需要时使用工具。")
agent_good = create_agent(model=model, tools=[query_product_price],
    system_prompt="你是一个助手，需要时使用工具。")

print("【坏描述结果】")
rprint(agent_bad.invoke({"messages": [HumanMessage(content="iPhone 16多少钱？")]}))
print("\n\n【好描述结果】")
rprint(agent_good.invoke({"messages": [HumanMessage(content="iPhone 16多少钱？")]}))

In [ ]:
# 案例 1：System prompt 引导工具使用
from langchain_core.messages import HumanMessage
from rich import print as rprint

# 问题场景：Agent 有时会"凭记忆"直接回答，跳过工具调用
# 解决：在 system prompt 中给出明确指令

@tool
def get_stock_price(ticker: str) -> str:
    """查询股票实时价格。"""
    prices = {"AAPL": "$180", "TSLA": "$250", "GOOGL": "$140"}
    return prices.get(ticker.upper(), "未找到")

# ❌ 模糊的 system prompt —— 模型可能直接编造
agent_weak_prompt = create_agent(
    model=model, tools=[get_stock_price],
    system_prompt="你是一个助手。",
)

# ✅ 明确的 system prompt —— 强制要求调用工具
agent_strong_prompt = create_agent(
    model=model, tools=[get_stock_price],
    system_prompt=(
        "你是一个金融数据助手。\n"
        "重要规则：\n"
        "- 任何涉及股价、股票信息的查询，必须先调用 get_stock_price 工具\n"
        "- 只使用工具返回的真实数据回答，绝不编造价格\n"
        "- 如果工具未返回数据，直接告知用户'暂无数据'"
    ),
)

print("【弱 Prompt】")
resp = agent_weak_prompt.invoke({"messages": [HumanMessage(content="苹果股价多少？")]})
has_tool_call = any(
    hasattr(m, "tool_calls") and m.tool_calls
    for m in resp["messages"]
)
print(f"  是否调用了工具: {'✅ 是' if has_tool_call else '❌ 否（可能编造了数据）'}")

print("\n【强 Prompt】")
resp = agent_strong_prompt.invoke({"messages": [HumanMessage(content="苹果股价多少？")]})
has_tool_call = any(
    hasattr(m, "tool_calls") and m.tool_calls
    for m in resp["messages"]
)
print(f"  是否调用了工具: {'✅ 是' if has_tool_call else '❌ 否'}")


# 案例 2：参数 default 值避免调用失败
from pydantic import BaseModel, Field

class WeatherInputWithDefault(BaseModel):
    city: str = Field(description="城市名称")
    unit: str = Field(
        default="celsius",  # ← 关键：给可选参数设默认值
        description="温度单位，celsius（摄氏）或 fahrenheit（华氏），默认为 celsius"
    )

@tool(args_schema=WeatherInputWithDefault)
def get_weather_v2(city: str, unit: str = "celsius") -> str:
    """查询天气。"""
    return f"{city}: 25°{'C' if unit == 'celsius' else 'F'}"

agent_default = create_agent(
    model=model, tools=[get_weather_v2],
    system_prompt="你是一个天气助手。",
)

# 用户没有指定温度单位 → default 值让调用仍然成功
resp = agent_default.invoke({"messages": [HumanMessage(content="北京天气怎么样？")]})
print("\n【参数默认值】用户未指定单位，但调用成功（使用默认 celsius）")
for msg in resp["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")

### 4.3 Why Agents Choose Wrong Tools

**常见原因**：

| # | 原因 | 示例 |
|---|------|------|
| 1 | 多个工具描述过于相似 | `search_news` 和 `search_wiki` 都写"搜索信息"，模型随机选 |
| 2 | 参数名在不同工具间含义模糊 | 工具 A 的 `query` 是"人名"，工具 B 的 `query` 是"城市名" |
| 3 | 工具粒度过细 | 拆成 N 个功能几乎一样的工具，模型反而困惑 |
| 4 | 描述中缺少"反例" | 没说**什么时候不该用**这个工具 |

**演示：容易混淆的工具 → 如何修正**

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich import print as rprint


# ---------- ❌ 坏示例：两个工具描述几乎一样 ----------
@tool
def get_stock_price(ticker: str) -> str:
    """查询价格。"""
    return f"{ticker} 股价: $150"


@tool
def get_house_price(address: str) -> str:
    """查询价格。"""
    return f"{address} 房价: $500,000"


# ---------- ✅ 好示例：描述写出明确的适用对象和场景 ----------
class StockInput(BaseModel):
    ticker: str = Field(description="股票代码，例如 AAPL、TSLA、600519")


@tool(args_schema=StockInput)
def get_stock_price_v2(ticker: str) -> dict:
    """查询美股或A股的最新股价和市场数据。仅用于股票、上市公司相关查询，不适用于房价查询。"""
    return {"ticker": ticker.upper(), "price": 150.00, "currency": "USD"}


class HouseInput(BaseModel):
    address: str = Field(description="房产地址，例如小区名、街道名")


@tool(args_schema=HouseInput)
def get_house_price_v2(address: str) -> dict:
    """查询指定小区或地址的房产均价（元/平米）。仅用于房产、住宅、租房相关查询，不适用于股票查询。"""
    return {"address": address, "avg_price_per_sqm": 50000, "currency": "CNY"}


# 对比测试
agent_bad = create_agent(model=model, tools=[get_stock_price, get_house_price],
    system_prompt="你是一个助手，遇到价格问题使用工具查询。")
agent_good = create_agent(model=model, tools=[get_stock_price_v2, get_house_price_v2],
    system_prompt="你是一个助手，遇到价格问题使用工具查询。")

print("【坏描述 —— 问股票却可能调到房价】")
rprint(agent_bad.invoke({"messages": [HumanMessage(content="特斯拉股价多少？")]}))
print("\n\n【好描述 —— 准确匹配】")
rprint(agent_good.invoke({"messages": [HumanMessage(content="特斯拉股价多少？")]}))

In [ ]:
# 案例 1：参数 description 用具体示例而非抽象概念
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from rich import print as rprint

# ❌ 坏：参数描述太抽象
class SearchInputBad(BaseModel):
    query: str = Field(description="搜索内容")  # 模型不知道怎么填

# ✅ 好：参数描述有具体示例
class SearchInputGood(BaseModel):
    query: str = Field(description="搜索关键词。例如：'2025年诺贝尔奖得主'、'iPhone 16 发布会日期'")

@tool(args_schema=SearchInputGood)
def web_search_v2(query: str) -> str:
    """搜索互联网获取最新信息。"""
    return f"搜索结果: {query}"


# 案例 2：功能相近的工具合并，通过参数区分行为

# ❌ 坏：三个几乎一样的工具，模型容易选错
@tool
def search_news(query: str) -> str:
    """搜索新闻。"""
    return f"新闻: {query}"

@tool
def search_blogs(query: str) -> str:
    """搜索博客。"""
    return f"博客: {query}"

@tool
def search_papers(query: str) -> str:
    """搜索论文。"""
    return f"论文: {query}"


# ✅ 好：合并为一个工具，用参数区分来源
class UnifiedSearchInput(BaseModel):
    query: str = Field(description="搜索关键词")
    source: str = Field(
        default="web",
        description="搜索来源：'news'（新闻）、'blogs'（博客）、'papers'（学术论文）、'web'（综合）"
    )

@tool(args_schema=UnifiedSearchInput)
def unified_search(query: str, source: str = "web") -> str:
    """统一搜索入口，支持新闻、博客、论文等多种来源。"""
    results = {
        "news": f"[新闻] {query} 相关新闻 3 条",
        "blogs": f"[博客] {query} 相关博文 5 篇",
        "papers": f"[论文] {query} 相关论文 2 篇",
        "web": f"[综合] {query} 搜索结果 10 条",
    }
    return results.get(source, results["web"])


# 对比：合并后 Agent 只需维护 1 个工具，选择负担大幅降低
agent_merged = create_agent(
    model=model, tools=[unified_search],
    system_prompt="你是一个搜索助手。",
)
resp = agent_merged.invoke({"messages": [HumanMessage(content="搜索关于AI的学术论文")]})
print("【合并工具效果】")
for msg in resp["messages"]:
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")

### 4.4 How to Know When an Agent Finishes

**问题**：Agent 在一次 `invoke` 中可能进行多轮工具调用（调用工具 → 收到结果 → 再调用工具 → …），如何判断它已经完成所有工具调用、给出了最终回答？

**判断方式**：

| 方法 | 说明 |
|------|------|
| 检查 `AIMessage` 是否包含 `tool_calls` | 如果最后一条 AI 消息**没有** tool_calls，说明模型认为不需要再调工具了 |
| 使用 `agent.stream()` 流式监听 | 实时观察每一步：tool_call 发起 → 工具执行 → 最终回复 |
| 检查消息列表末尾 | `response["messages"][-1]` 如果是 AIMessage 且 `.content` 非空，通常就是最终回答 |

**演示：用 stream 实时观察 Agent 的每一步决策**

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from rich import print as rprint

# 复用 2.3 节中定义的工具（get_weather、get_exchange_rate、calculate）
# 如果当前 kernel 未执行 2.3 节，请先执行 2.3 节的工具定义代码

print("【实时观察 Agent 的每一步】")
print("-" * 50)

for chunk in agent.stream({
    "messages": [
        HumanMessage(content="查一下东京的天气，然后把 1000 美元换成日元")
    ]
}):
    # chunk 是一个 {node_name: {messages: [...]}} 的字典
    for node_name, node_output in chunk.items():
        messages = node_output.get("messages", [])
        for msg in messages:
            if isinstance(msg, AIMessage):
                if msg.tool_calls:
                    # 模型决定调用工具
                    for tc in msg.tool_calls:
                        print(f"[use tool] {tc['name']}({tc['args']})")
                elif msg.content:
                    # 无 tool_calls → 最终回答
                    print(f"\n[final answer]\n{msg.content}")
            elif isinstance(msg, ToolMessage):
                # 工具返回结果
                preview = str(msg.content)[:100]
                print(f"[tool returned] {preview}...")


In [ ]:
# 案例 1：检查消息类型判断 Agent 是否完成
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from rich import print as rprint

# 复用 2.3 节创建的 agent
resp = agent.invoke({
    "messages": [HumanMessage(content="北京天气怎么样？")]
})

print("【逐条分析消息列表】")
for i, msg in enumerate(resp["messages"]):
    msg_type = type(msg).__name__
    if isinstance(msg, AIMessage):
        has_tc = bool(msg.tool_calls)
        has_content = bool(msg.content)
        status = "🔧 调用工具中..." if has_tc else ("✅ 最终回答" if has_content else "⏳ 等待中")
        print(f"  [{i}] {msg_type} | tool_calls={has_tc} | content_len={len(msg.content) if msg.content else 0} | {status}")
    elif isinstance(msg, ToolMessage):
        print(f"  [{i}] {msg_type} | name={msg.name} | 工具已返回结果")
    elif isinstance(msg, HumanMessage):
        print(f"  [{i}] {msg_type} | 用户输入")
    else:
        print(f"  [{i}] {msg_type}")


# 案例 2：编程方式判断 Agent 是否真正完成
def is_agent_finished(response: dict) -> tuple[bool, str]:
    """
    判断 Agent 是否已完成所有工具调用并给出最终回答。
    返回 (是否完成, 原因说明)。
    """
    messages = response.get("messages", [])
    if not messages:
        return False, "消息列表为空"

    last_msg = messages[-1]

    # 检查最后一条消息：必须是 AIMessage，无 tool_calls，有实际内容
    if isinstance(last_msg, AIMessage):
        if last_msg.tool_calls:
            return False, f"最后一条消息仍有 tool_calls: {[tc['name'] for tc in last_msg.tool_calls]}"
        if not last_msg.content:
            return False, "最后一条 AIMessage 内容为空"
        return True, "Agent 已完成并给出最终回答"

    # 如果最后一条是 ToolMessage，说明工具执行完了但模型还没生成最终回答（异常状态）
    if isinstance(last_msg, ToolMessage):
        return False, f"最后一条是工具返回（{last_msg.name}），模型尚未生成最终回答"

    return False, f"最后一条是 {type(last_msg).__name__}，非预期类型"


finished, reason = is_agent_finished(resp)
print(f"\n【完成判定】{reason}")
print(f"  最终回答: {resp['messages'][-1].content[:100] if finished else 'N/A'}...")


# 案例 3：兜底截断 —— 防止模型在最终回答中仍带 tool_calls
def safe_invoke(agent, input_msg: str, max_retries: int = 3) -> str:
    """
    安全调用 Agent：自动检测并处理异常状态。
    - 未完成 → 重试
    - 超限 → 强制要求给出回答
    """
    for attempt in range(max_retries):
        resp = agent.invoke({
            "messages": [HumanMessage(content=input_msg)]
        })
        finished, reason = is_agent_finished(resp)

        if finished:
            return resp["messages"][-1].content

        print(f"[重试 {attempt + 1}/{max_retries}] {reason}")
        # 追问模型，要求给出最终回答
        input_msg = "请基于已有信息直接给出最终回答，不要再调用工具。"

    # 最终兜底：返回最后一条有内容的消息
    for msg in reversed(resp["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return f"[兜底截断] {msg.content}"
    return "[错误] 未能获取有效回答"

result = safe_invoke(agent, "查一下北京天气")
print(f"\n【安全调用结果】\n{result[:200]}...")

### 4.5 How to Limit Tool Calls

**问题**：Agent 有时会陷入"调用工具 -> 不满意 -> 再调用 -> 再调用"的循环，或对简单问题反复调用工具，消耗大量 token 和时间。

下面把每种方式拆成独立代码 cell。先运行公共准备 cell，再分别运行每个方式的示例。每个示例都会打印工具调用轨迹和拦截统计。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware, before_model
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from rich import print as rprint

load_dotenv(".env")

DEMO_TOOLS = [get_weather, get_exchange_rate, calculate]
BLOCK_PREFIX = "Tool call limit exceeded"
MAX_TOOL_CALLS = 4
RECURSION_LIMIT = MAX_TOOL_CALLS * 3 + 5

limit_demo_model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)


def tool_messages(messages) -> list[ToolMessage]:
    """取出所有 ToolMessage，包括真实工具结果和被限流拦截的结果。"""
    return [m for m in messages if isinstance(m, ToolMessage)]


def count_successful_tool_results(messages) -> int:
    """统计真正执行成功的工具结果数。"""
    return sum(
        not str(m.content).startswith(BLOCK_PREFIX)
        for m in tool_messages(messages)
    )


def count_blocked_tool_results(messages) -> int:
    """统计被 ToolCallLimitMiddleware 拦截的工具调用数。"""
    return sum(
        str(m.content).startswith(BLOCK_PREFIX)
        for m in tool_messages(messages)
    )


def print_tool_trace(response: dict, *, max_success: int | None = None) -> None:
    """打印模型请求了哪些工具，以及哪些工具真正执行或被拦截。"""
    messages = response["messages"]
    print("\n[工具轨迹]")
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"  请求工具 -> {tool_call['name']}({tool_call['args']})")
        elif isinstance(msg, ToolMessage):
            blocked = str(msg.content).startswith(BLOCK_PREFIX)
            status = "已拦截" if blocked else "已执行"
            preview = str(msg.content).replace("\n", " ")[:90]
            print(f"  工具结果 <- {msg.name}: {status} | {preview}")

    success = count_successful_tool_results(messages)
    blocked = count_blocked_tool_results(messages)
    print("\n[统计]")
    if max_success is None:
        print(f"  成功执行：{success}")
    else:
        print(f"  成功执行：{success}/{max_success}")
    print(f"  被拦截：{blocked}")


def print_final_answer(response: dict, limit: int = 900) -> None:
    """打印最终回答，避免调试输出淹没正文。"""
    print("\n[最终回答]")
    rprint(response["messages"][-1].content[:limit])

#### Method 1: recursion_limit (Hard Limit)

用于限制 LangGraph 图节点执行次数。它是最后一道硬兜底，触发后会中断整次运行。

In [ ]:
print("【方式 1：recursion_limit 图执行兜底】")

agent_recursion_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt="你是一个助手。需要信息时可以调用工具，工具返回后再回答。",
)

try:
    agent_recursion_limited.invoke(
        {"messages": [HumanMessage(content="查北京天气，然后给出最终回答")]},
        config={"recursion_limit": 2},
    )
except GraphRecursionError as e:
    print("\n[拦截生效]")
    print("  recursion_limit=2 太低，任务至少需要 model -> tools -> model。")
    print(f"  LangGraph 已终止执行：{type(e).__name__}")

#### Method 2: System Prompt Guidance

通过规则告诉模型不要重复调用相同参数的工具。这种方式成本最低，但依赖模型遵守指令。

In [ ]:
print("【方式 2：System Prompt 引导少调用】")

system_prompt="""你是一个助手，可以使用工具查询信息。
规则：
1. 入参相同时，只调用一次工具
2. 不要追问确认，直接回答
"""
agent_prompt_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个助手，可以使用工具查询信息。\n"
        "规则：\n"
        "1. 入参相同时，只调用一次工具\n"
        "2. 不要追问确认，直接回答。"
    ),
)

response = agent_prompt_limited.invoke(
    {"messages": [HumanMessage(content="帮我查询10次北京的天气，每次查询的结果都给我")]},
    config={"recursion_limit": RECURSION_LIMIT},
)
rprint(response)
print_tool_trace(response)
print_final_answer(response)

#### Method 3: ToolCallLimitMiddleware

在工具执行层限制成功调用次数。即使模型一次性请求 10 次工具，也只允许前 N 次真正执行。

In [ ]:
print("【方式 3：ToolCallLimitMiddleware 执行层拦截】")

agent_tool_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个工具限流演示助手。\n"
        "如果用户要求查询 N 次，请发起 N 次 get_weather 工具调用。\n"
        "如果工具返回限流错误，请在最终回答里标出哪些次数被拦截。"
    ),
    middleware=[ToolCallLimitMiddleware(run_limit=MAX_TOOL_CALLS, exit_behavior="continue")],
)

response = agent_tool_limited.invoke(
    {"messages": [HumanMessage(content="请调用 get_weather 查询北京天气 10 次，并逐次列出结果")]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print_tool_trace(response, max_success=MAX_TOOL_CALLS)
print_final_answer(response)

#### Best Practice: Three-Layer Defense

用 prompt 减少不必要调用，用 middleware 做工具层限流，用 `recursion_limit` 防止图执行失控。

In [ ]:
print("【最佳实践：Prompt 引导 + Middleware 限流 + recursion_limit 兜底】")

BEST_MAX_TOOL_CALLS = 6
BEST_WARN_REMAINING = 2
BEST_RECURSION_LIMIT = BEST_MAX_TOOL_CALLS * 3 + 5


def has_limit_notice(messages, prefix: str) -> bool:
    """避免同一种限制提醒被重复注入到上下文里。"""
    return any(
        isinstance(m, HumanMessage)
        and isinstance(m.content, str)
        and m.content.startswith(prefix)
        for m in messages
    )


@before_model
def gentle_limit(state, runtime):
    """接近工具上限时提醒模型收束。真正的硬拦截交给 ToolCallLimitMiddleware。"""
    messages = state.get("messages", [])
    success_count = count_successful_tool_results(messages)
    remaining = BEST_MAX_TOOL_CALLS - success_count

    if remaining <= 0 and not has_limit_notice(messages, "[工具调用限制]"):
        return {"messages": messages + [HumanMessage(content=(
            f"[工具调用限制] 已成功执行 {success_count} 次工具，达到上限 {BEST_MAX_TOOL_CALLS}。"
            "必须基于已有结果回答，不要再调用工具。"
        ))]}

    if remaining == BEST_WARN_REMAINING and not has_limit_notice(messages, "[工具调用提醒]"):
        return {"messages": messages + [HumanMessage(content=(
            f"[工具调用提醒] 还可成功调用 {remaining} 次工具。请尽快收束回答。"
        ))]}

    return None


agent_best_practice = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个智能助手。\n"
        "规则：\n"
        "1. 每个子任务最多调用 1 次工具。\n"
        "2. 相同参数的工具调用不要重复执行。\n"
        "3. 如果用户要求重复查询同一参数，只调用 1 次工具，并把这一次结果重复列出用户要求的次数。\n"
        "4. 工具返回结果后必须直接整合回答，不要追问确认。"
    ),
    middleware=[
        gentle_limit,
        ToolCallLimitMiddleware(run_limit=BEST_MAX_TOOL_CALLS, exit_behavior="continue"),
    ],
    checkpointer=InMemorySaver(),
)

response = agent_best_practice.invoke(
    {"messages": [HumanMessage(content="反复查询北京天气 10 次，每次都告诉我结果")]},
    config={
        "recursion_limit": BEST_RECURSION_LIMIT,
        "configurable": {"thread_id": "tool-limit-best-practice-demo"},
    },
)

print_tool_trace(response, max_success=BEST_MAX_TOOL_CALLS)
print_final_answer(response)